In [2]:
from pathlib import Path
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from pinecone import Pinecone, ServerlessSpec

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()


True

In [4]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

#Loading coffe docs
HTML_DIR = Path(r"coffee_pages")

In [5]:
def get_html_text(file_path: Path) -> str:
    with open(file_path, "r", encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator=" ", strip=True)

# Load coffee HTML documents
coffee_docs = []
for idx, fp in enumerate(HTML_DIR.glob("*.html")):
    coffee_docs.append({
        "id": f"coffee_{idx}",
        "text": get_html_text(fp),
        "file_name": fp.name,
        "created_at": "2025-12-10"
    })

In [6]:
simple_docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis.",
     "file_name": "doc1.txt", "created_at": "2025-12-10"},
    {"id": "doc2", "text": "Pinecone is a vector database for semantic search.",
     "file_name": "doc2.txt", "created_at": "2025-12-10"},
    {"id": "doc3", "text": "Spark enables distributed data processing.",
     "file_name": "doc3.txt", "created_at": "2025-12-10"}
]

docs = coffee_docs + simple_docs
texts = [d["text"] for d in docs]

In [7]:
# TF-IDF Vectors we will use
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(texts)

In [8]:
vocab_size = len(tfidf.vocabulary_)
print("Vocabulary size:", vocab_size)

Vocabulary size: 938


In [9]:
def row_to_sparse_values(csr_row):
    coo = csr_row.tocoo()
    return {
        "indices": coo.col.tolist(),
        "values": [float(v) for v in coo.data.tolist()]
    }

In [10]:
pc = Pinecone(api_key=pinecone_api_key)
INDEX_NAME = "coffee-sparse-index"

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        vector_type="sparse",   
        metric="dotproduct",     
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

# target the correct host
desc = pc.describe_index(name=INDEX_NAME)
index = pc.Index(host=desc.host)

In [11]:
BATCH = 100
payload = []

for i, d in enumerate(docs):
    sparse_vec = row_to_sparse_values(X[i])

    payload.append({
        "id": d["id"],
        "sparse_values": sparse_vec,
        "metadata": {
            "file_name": d["file_name"],
            "created_at": d["created_at"],
            "chunk_text": d["text"][:500]   # preview optional
        }
    })

    if len(payload) == BATCH:
        index.upsert(vectors=payload)
        payload.clear()

# final leftover batch
if payload:
    index.upsert(vectors=payload)

print(f"Upserted {len(docs)} sparse vectors into '{INDEX_NAME}'.")


Upserted 18 sparse vectors into 'coffee-sparse-index'.


In [12]:
# Getting the Output
def lexical_query(query_text: str, top_k: int = 5, include_text=False):
    q = tfidf.transform([query_text])
    sparse_q = row_to_sparse_values(q)

    res = index.query(
        sparse_vector=sparse_q,
        top_k=top_k,
        include_values=False,
        include_metadata=True
    )

    out = []
    for m in res.get("matches", []):
        item = {
            "id": m["id"],
            "score": m["score"],
            "file_name": m["metadata"].get("file_name")
        }
        if include_text:
            item["preview"] = m["metadata"].get("chunk_text", "")
        out.append(item)

    return out

In [13]:
results = lexical_query("Which is best coffee for health? and what is DB?", top_k=3, include_text=True)
results

[{'id': 'coffee_1',
  'score': 0.222563863,
  'file_name': '02_masala_coffee_spiced_indian_coffee.html',
  'preview': 'Masala Coffee (Spiced Indian Coffee) Masala Coffee (Spiced Indian Coffee) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursing, or taking medications should consult a qualified professional before trying new herbs or routines. Overview Masala coffee infuses classic Indian spices into brewed coffee. Cardamom is common, with cinnamon, clove, nutmeg, and black pepper addin'},
 {'id': 'coffee_0',
  'score': 0.178301364,
  'file_name': '01_ashwagandha_coffee_adaptogenic_latte.html',
  'preview': 'Ashwagandha Coffee (Adaptogenic Latte) Ashwagandha Coffee (Adaptogenic Latte) Updated on August 17, 2025 Disclaimer: This page shares general culinary and cultural information. It is not medical advice. Individuals with health conditions, pregnant or nursi

# Hybrid Search

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import numpy as np
from pinecone import Pinecone, ServerlessSpec

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")


In [3]:
HTML_DIR = Path("coffee_pages")

def get_html_text(path: Path) -> str:
    with open(path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    return soup.get_text(separator=" ", strip=True)

coffee_docs = []
for idx, fp in enumerate(HTML_DIR.glob("*.html")):
    coffee_docs.append({
        "id": f"coffee_{idx}",
        "text": get_html_text(fp),
        "file_name": fp.name,
        "created_at": "2025-12-10"
    })

In [4]:
simple_docs = [
    {"id": "doc1", "text": "Pandas is a Python library for data analysis.",
     "file_name": "doc1.txt", "created_at": "2025-12-10"},

    {"id": "doc2", "text": "Pinecone is a vector database for semantic search.",
     "file_name": "doc2.txt", "created_at": "2025-12-10"},

    {"id": "doc3", "text": "Spark enables distributed data processing.",
     "file_name": "doc3.txt", "created_at": "2025-12-10"}
]

docs = coffee_docs + simple_docs
texts = [d["text"] for d in docs]


In [5]:
tfidf = TfidfVectorizer()

X = tfidf.fit_transform(texts)   

In [6]:
def csr_row_to_pinecone(csr_row):
    coo = csr_row.tocoo()
    return {
        "indices": coo.col.tolist(),
        "values": [float(v) for v in coo.data.tolist()]
    }


In [7]:
# Dense Embeddings
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")
dense = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)


In [8]:
pc = Pinecone(api_key=pinecone_api_key)
HYBRID_INDEX = "hybridsearchcoffeedocs"

pc.create_index(
    name=HYBRID_INDEX,
    dimension=384,               # dense vector dimension
    metric="dotproduct",     
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

# connect
desc = pc.describe_index(HYBRID_INDEX)
hy = pc.Index(host=desc.host)


In [9]:
vectors_hybrid = []

for i, d in enumerate(docs):
    sparse_part = csr_row_to_pinecone(X[i])

    vectors_hybrid.append({
        "id": d["id"],
        "values": dense[i].tolist(),     # dense 384-d embedding
        "sparse_values": sparse_part,    # TF-IDF sparse embedding
        "metadata": {
            "file_name": d["file_name"],
            "created_at": d["created_at"],
            "text_preview": d["text"][:300]
        }
    })

hy.upsert(vectors=vectors_hybrid)
print("Hybrid upsert completed!")

Hybrid upsert completed!


In [11]:
query = "what is indian coffee and what is vecotr db"

q_dense = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].tolist()

res_dense = hy.query(
    vector=q_dense,
    top_k=3,
    include_metadata=True
)

res_dense

QueryResponse(matches=[{'id': 'coffee_3',
 'metadata': {'created_at': '2025-12-10',
              'file_name': '04_south_indian_filter_coffee_with_chicory.html',
              'text_preview': 'South Indian Filter Coffee with Chicory South '
                              'Indian Filter Coffee with Chicory Updated on '
                              'August 17, 2025 Disclaimer: This page shares '
                              'general culinary and cultural information. It '
                              'is not medical advice. Individuals with health '
                              'conditions, pregnant or nursing, or taking '
                              'medications should consult a'},
 'score': 0.58886385,
 'values': []}, {'id': 'coffee_11',
 'metadata': {'created_at': '2025-12-10',
              'file_name': '12_coconut_milk_coffee_south_coastal_style.html',
              'text_preview': 'Coconut Milk Coffee (South Coastal Style) '
                              'Coconut Milk Coffee (

In [27]:
query = "which indian coffee is sweet and what is vector db"

alpha = 0.5

#Dense embeddings
q_dense = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].tolist()
q_dense_w = (np.array(q_dense) * alpha).tolist()

#Sparse embeddings
q_row = tfidf.transform([query])
q_sparse = csr_row_to_pinecone(q_row[0])
q_sparse_w = {
    "indices": q_sparse["indices"],
    "values": [v * (1 - alpha) for v in q_sparse["values"]]
}

res_hybrid = hy.query(
    vector=q_dense_w,
    sparse_vector=q_sparse_w,
    top_k=5,
    include_metadata=True
)

res_hybrid

QueryResponse(matches=[{'id': 'coffee_3',
 'metadata': {'created_at': '2025-12-10',
              'file_name': '04_south_indian_filter_coffee_with_chicory.html',
              'text_preview': 'South Indian Filter Coffee with Chicory South '
                              'Indian Filter Coffee with Chicory Updated on '
                              'August 17, 2025 Disclaimer: This page shares '
                              'general culinary and cultural information. It '
                              'is not medical advice. Individuals with health '
                              'conditions, pregnant or nursing, or taking '
                              'medications should consult a'},
 'score': 0.366364121,
 'values': []}, {'id': 'coffee_1',
 'metadata': {'created_at': '2025-12-10',
              'file_name': '02_masala_coffee_spiced_indian_coffee.html',
              'text_preview': 'Masala Coffee (Spiced Indian Coffee) Masala '
                              'Coffee (Spiced Indian Co